# Evaluador FOM-5 v2 local

Evaluación **offline y reproducible** de formulaciones de optimización generadas por `Qwen/Qwen2.5-1.5B-Instruct`.

Este notebook:

- **no vuelve a generar** las respuestas del modelo base;
- reutiliza `resultados_fom5.csv`;
- ejecuta el judge **localmente en Google Colab**;
- usa `Qwen/Qwen3-14B` cuantizado a 4 bits como judge;
- calibra primero cinco casos con errores ya revisados manualmente;
- solo permite ejecutar las 32 evaluaciones si la calibración pasa;
- guarda checkpoint después de cada caso;
- mantiene los 30 problemas como benchmark oficial y el caso oftalmológico solo como diagnóstico histórico.

> Recomendación: abrir un entorno de Colab limpio con GPU T4 antes de ejecutar este notebook.


In [ ]:
# Instalar solo lo necesario.
# No actualizamos pandas ni requests porque Colab ya trae versiones compatibles.
!pip -q install -U "transformers>=4.51,<5" accelerate bitsandbytes tqdm


In [ ]:
import os
import io
import re
import gc
import json
import time
import gzip
import base64
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from google.colab import files

warnings.filterwarnings("ignore")

assert torch.cuda.is_available(), (
    "Este notebook está diseñado para GPU. "
    "En Colab: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU."
)

gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("GPU:", gpu_name)
print(f"VRAM: {gpu_gb:.1f} GB")
print("PyTorch:", torch.__version__)


In [ ]:
# Golds auditados embebidos: 30 benchmark + 1 diagnóstico histórico.
GOLD = json.loads(
    gzip.decompress(
        base64.b64decode('...')
    ).decode("utf-8")
)

GOLD_LOOKUP = {item["id"]: item for item in GOLD}

print("Problemas oficiales:", sum(x["id"].startswith("opt_") for x in GOLD))
print("Diagnósticos adicionales:", sum(not x["id"].startswith("opt_") for x in GOLD))


## 1. Cargar las respuestas ya generadas

Sube el archivo `resultados_fom5.csv` obtenido con el notebook del baseline.  
La columna esencial es `candidate`; este notebook no vuelve a ejecutar Qwen2.5-1.5B.


In [ ]:
uploaded = files.upload()

csv_candidates = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_candidates:
    raise ValueError("Debes subir resultados_fom5.csv")

csv_name = csv_candidates[0]
old_df = pd.read_csv(io.BytesIO(uploaded[csv_name]))

required_columns = {"id", "variant", "candidate"}
missing = required_columns - set(old_df.columns)
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {sorted(missing)}")

unknown_ids = sorted(set(old_df["id"]) - set(GOLD_LOOKUP))
if unknown_ids:
    raise ValueError(f"Hay IDs sin gold auditado: {unknown_ids}")

print("Archivo:", csv_name)
print("Filas:", len(old_df))
display(old_df[["id", "variant"]].head())


## 2. Carpeta de salida persistente

Por defecto se usa Google Drive para que un reinicio de Colab no elimine el checkpoint.  
Si no quieres montar Drive, cambia `USE_DRIVE = False`.


In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/fom5_local")
else:
    OUT_DIR = Path("/content/fom5_local")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Salida:", OUT_DIR)


## 3. Cargar judge local

Judge fijo: `Qwen/Qwen3-14B`, cuantizado NF4 a 4 bits.

El judge se usa únicamente para **evaluación offline**. El modelo del sistema estudiado sigue siendo `Qwen/Qwen2.5-1.5B-Instruct`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

JUDGE_MODEL = "Qwen/Qwen3-14B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_MODEL,
    use_fast=True,
)

model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.eval()

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print("Judge:", JUDGE_MODEL)
print(f"VRAM asignada: {allocated:.2f} GB")
print(f"VRAM reservada: {reserved:.2f} GB")


## 4. Protocolo FOM-5 v2

La salida del judge se mantiene deliberadamente corta. El modelo debe decidir:

- si existe una formulación matemática;
- `met / partial / missing / wrong` para cada requisito auditado;
- cinco scores brutos de 0 a 5;
- errores fatales breves.

El código aplica después los topes deterministas de FOM-5 v2.


In [ ]:
SYSTEM_JUDGE = r"""
Eres un juez estricto de formulaciones de Investigación de Operaciones.

TAREA DEL CANDIDATO:
Traducir un problema PL/MILP desde lenguaje natural a una formulación matemática.
Debe formular, NO resolver numéricamente.

REGLAS:
1. Evalúa solo lo escrito. No completes ecuaciones faltantes.
2. Acepta formulaciones matemáticamente equivalentes al gold aunque cambien símbolos o técnica.
3. Una restricción esencial omitida, invertida o aplicada a la variable equivocada es missing/wrong.
4. Una explicación, algoritmo, código o solución numérica no sustituye una formulación matemática.
5. Si falta una variable decisional esencial, decision_domains <= 2.
6. Si el objetivo tiene el sentido correcto pero omite un término esencial, objective <= 2.
7. Si no hay objetivo matemático, objective <= 1.
8. Si no existe formulación matemática identificable, has_formulation=false.
9. Contradicciones entre ecuaciones penalizan constraints_logic y algebra_validity.
10. Si formula correctamente pero además resuelve la instancia, generalization <= 4.
11. Si sustituye la formulación por una solución numérica, has_formulation=false.
12. No premies prosa ni castigues notación distinta si es coherente.

ESCALA 0-5:
0 ausente/totalmente inválido
1 falla fundamental
2 estructura reconocible con error esencial
3 mayormente correcta, pero falta/falla una pieza material
4 correcta salvo detalle menor
5 completa y matemáticamente correcta/equivalente

Estados de requisito permitidos:
met, partial, missing, wrong

Devuelve SOLO JSON válido. Sin markdown y sin explicaciones largas.
"""

DIMENSIONS = [
    "decision_domains",
    "objective",
    "constraints_logic",
    "algebra_validity",
    "generalization",
]

def judge_prompt(item, candidate):
    req_lines = "\n".join(
        f'{r["id"]} [{r["kind"]}]: {r["text"]}'
        for r in item["requirements"]
    )

    req_template = ", ".join(
        f'"{r["id"]}":"met|partial|missing|wrong"'
        for r in item["requirements"]
    )

    return f"""
ENUNCIADO:
{item["problem"]}

REQUISITOS:
{req_lines}

GOLD DE REFERENCIA:
{item["reference_model"]}

RESPUESTA CANDIDATA:
{candidate}

Responde exactamente con esta estructura:
{{
  "has_formulation": true,
  "requirements": {{{req_template}}},
  "scores": {{
    "decision_domains": 0,
    "objective": 0,
    "constraints_logic": 0,
    "algebra_validity": 0,
    "generalization": 0
  }},
  "fatal_errors": ["error breve si corresponde"]
}}
"""


In [ ]:
MAX_INPUT_TOKENS = 12000
MAX_NEW_TOKENS = 600

def _chat_text(messages):
    # Qwen3 permite desactivar thinking explícito.
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def extract_json(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    start = text.find("{")
    if start < 0:
        raise ValueError("No se encontró inicio de JSON.")

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i+1])

    raise ValueError("JSON incompleto.")

def validate_parsed(item, parsed):
    if not isinstance(parsed, dict):
        raise ValueError("La respuesta parseada no es un objeto.")

    if "has_formulation" not in parsed:
        raise ValueError("Falta has_formulation.")

    reqs = parsed.get("requirements")
    if not isinstance(reqs, dict):
        raise ValueError("requirements debe ser un objeto {R1: estado, ...}.")

    expected = {r["id"] for r in item["requirements"]}
    got = set(reqs)

    if expected != got:
        raise ValueError(
            f"IDs de requisitos incorrectos. Esperados={expected}; recibidos={got}"
        )

    allowed = {"met", "partial", "missing", "wrong"}
    bad_status = {
        rid: status
        for rid, status in reqs.items()
        if str(status).lower().strip() not in allowed
    }
    if bad_status:
        raise ValueError(f"Estados inválidos: {bad_status}")

    scores = parsed.get("scores")
    if not isinstance(scores, dict):
        raise ValueError("Falta scores.")

    for dim in DIMENSIONS:
        if dim not in scores:
            raise ValueError(f"Falta score: {dim}")
        value = float(scores[dim])
        if not (0 <= value <= 5):
            raise ValueError(f"Score fuera de rango en {dim}: {value}")

    if "fatal_errors" not in parsed:
        parsed["fatal_errors"] = []

    return parsed

@torch.inference_mode()
def judge_local(item, candidate, max_retries=2):
    prompt = judge_prompt(item, candidate)
    messages = [
        {"role": "system", "content": SYSTEM_JUDGE},
        {"role": "user", "content": prompt},
    ]

    last_error = None
    last_raw = None

    for attempt in range(max_retries):
        local_messages = list(messages)
        if attempt > 0:
            local_messages.append({
                "role": "user",
                "content": (
                    "La salida anterior no pudo parsearse. "
                    "Repite el mismo juicio y devuelve SOLO el JSON solicitado, "
                    "completo y sin texto adicional."
                ),
            })

        text = _chat_text(local_messages)
        inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

        n_input = int(inputs["input_ids"].shape[1])
        if n_input > MAX_INPUT_TOKENS:
            raise ValueError(
                f"Prompt de {n_input} tokens supera MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}. "
                "No se truncó silenciosamente."
            )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        try:
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            generated = output[0, inputs["input_ids"].shape[1]:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            last_raw = raw

            parsed = validate_parsed(item, extract_json(raw))
            return parsed, raw, n_input, int(generated.shape[0])

        except Exception as e:
            last_error = str(e)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"Judge local falló tras {max_retries} intentos: {last_error}\n"
        f"Última salida: {last_raw}"
    )


## 5. Agregación determinista FOM-5 v2

Ponderaciones congeladas:

- decisiones y dominios: 15%
- objetivo: 20%
- restricciones/lógica: 35%
- validez algebraica: 15%
- generalización: 15%

Los estados de requisitos aplican topes deterministas sobre las dimensiones correspondientes.


In [ ]:
STATUS_VALUE = {
    "met": 1.0,
    "partial": 0.5,
    "missing": 0.0,
    "wrong": 0.0,
}

def safe_score(value):
    return max(0.0, min(5.0, float(value)))

def aggregate_v2(item, parsed):
    status = {
        rid: STATUS_VALUE[str(st).lower().strip()]
        for rid, st in parsed["requirements"].items()
    }

    by_kind = {"objective": [], "constraint": [], "domain": []}

    for req in item["requirements"]:
        by_kind[req["kind"]].append(
            status.get(req["id"], 0.0)
        )

    obj_cov = np.mean(by_kind["objective"]) if by_kind["objective"] else 1.0
    con_cov = np.mean(by_kind["constraint"]) if by_kind["constraint"] else 1.0
    dom_cov = np.mean(by_kind["domain"]) if by_kind["domain"] else 1.0

    raw = parsed["scores"]

    decision = safe_score(raw["decision_domains"])
    objective = min(safe_score(raw["objective"]), 5.0 * obj_cov)
    constraints = min(safe_score(raw["constraints_logic"]), 5.0 * con_cov)
    algebra = safe_score(raw["algebra_validity"])
    generalization = safe_score(raw["generalization"])

    if by_kind["domain"]:
        decision = min(decision, 5.0 * dom_cov)

    score = (
        0.15 * decision
        + 0.20 * objective
        + 0.35 * constraints
        + 0.15 * algebra
        + 0.15 * generalization
    )

    if not bool(parsed["has_formulation"]):
        algebra = min(algebra, 1.0)
        generalization = min(generalization, 1.0)

        score = min(
            1.5,
            0.15 * decision
            + 0.20 * objective
            + 0.35 * constraints
            + 0.15 * algebra
            + 0.15 * generalization
        )

    all_cov = np.mean(list(status.values())) if status else np.nan

    return {
        "score_fom5_v2": round(float(score), 3),
        "decision_domains_v2": float(decision),
        "objective_v2": float(objective),
        "constraints_logic_v2": float(constraints),
        "algebra_validity_v2": float(algebra),
        "generalization_v2": float(generalization),
        "requirement_coverage_v2": float(all_cov),
        "objective_coverage": float(obj_cov),
        "constraint_coverage": float(con_cov),
        "domain_coverage": float(dom_cov),
    }


## 6. Calibración obligatoria

Estos umbrales son **sanity checks**, no notas manuales oficiales.  
Los cinco casos tienen errores estructurales previamente inspeccionados. Si el judge no los detecta, no debe usarse para el benchmark completo.


In [ ]:
CALIBRATION_CASES = [
    ("opt_02", "baseline_direct_deterministic", 2.8,
     "variable incorrecta en restricciones y término de costo omitido"),
    ("opt_10", "baseline_direct_deterministic", 1.5,
     "no entrega una formulación matemática válida"),
    ("opt_14", "baseline_direct_deterministic", 1.5,
     "intenta resolver max-flow en vez de formularlo"),
    ("opt_16", "baseline_direct_deterministic", 1.8,
     "estructura multicommodity incorrecta"),
    ("diagnostic_oftalmo", "entrega1_historical_sampled", 2.5,
     "faltan apertura/linking/ponderación y hay contradicciones"),
]

calibration_outputs = {}
calibration_rows = []

for iid, variant, expected_max, reason in tqdm(
    CALIBRATION_CASES,
    desc="Calibración local"
):
    selected = old_df[
        (old_df["id"] == iid)
        & (old_df["variant"] == variant)
    ]

    if selected.empty:
        raise ValueError(f"No encontrado: {iid} / {variant}")

    rec = selected.iloc[0]
    item = GOLD_LOOKUP[iid]

    t0 = time.time()

    parsed, raw, in_tok, out_tok = judge_local(
        item,
        rec["candidate"],
    )

    elapsed = time.time() - t0
    agg = aggregate_v2(item, parsed)

    key = (iid, variant)

    evaluation = rec.to_dict()
    evaluation.update(agg)
    evaluation.update({
        "has_formulation_v2": bool(parsed["has_formulation"]),
        "fatal_errors_v2": json.dumps(parsed.get("fatal_errors", []), ensure_ascii=False),
        "requirements_v2": json.dumps(parsed["requirements"], ensure_ascii=False),
        "judge_scores_raw_v2": json.dumps(parsed["scores"], ensure_ascii=False),
        "judge_v2_raw": raw,
        "judge_v2_error": None,
        "judge_input_tokens": in_tok,
        "judge_output_tokens": out_tok,
        "judge_latency_sec": elapsed,
        "judge_model": JUDGE_MODEL,
    })

    calibration_outputs[key] = evaluation

    calibration_rows.append({
        "id": iid,
        "variant": variant,
        "score_v2": agg["score_fom5_v2"],
        "expected_max": expected_max,
        "passes": agg["score_fom5_v2"] <= expected_max,
        "coverage": agg["requirement_coverage_v2"],
        "seconds": round(elapsed, 1),
        "reason": reason,
        "fatal_errors": parsed.get("fatal_errors", []),
    })

calibration_df = pd.DataFrame(calibration_rows)

display(calibration_df[[
    "id",
    "score_v2",
    "expected_max",
    "passes",
    "coverage",
    "seconds",
    "fatal_errors",
]])

CALIBRATION_OK = bool(calibration_df["passes"].all())

print()
if CALIBRATION_OK:
    print("CALIBRACIÓN SUPERADA.")
    print("El judge queda congelado para esta corrida.")
else:
    print("CALIBRACIÓN NO SUPERADA.")
    print("NO ejecutes el benchmark completo con este judge.")

calibration_df.to_csv(
    OUT_DIR / "calibracion_fom5_local.csv",
    index=False,
    encoding="utf-8-sig",
)


## 7. Evaluación completa con checkpoint

La celda se detiene si la calibración no pasó.

Los cinco casos ya utilizados en calibración se reutilizan directamente porque fueron evaluados con **el mismo modelo, prompt, parámetros y agregación**. El resto se evalúa una sola vez.

Si Colab se interrumpe, vuelve a ejecutar esta celda: retomará desde el checkpoint almacenado en Drive.


In [ ]:
if not CALIBRATION_OK:
    raise RuntimeError(
        "La calibración no pasó. "
        "No se ejecutará el benchmark completo."
    )

CHECKPOINT = OUT_DIR / "resultados_fom5_local_checkpoint.csv"
FINAL_CSV = OUT_DIR / "resultados_fom5_local.csv"
FINAL_JSONL = OUT_DIR / "resultados_fom5_local.jsonl"

if CHECKPOINT.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT)
    rows = checkpoint_df.to_dict("records")

    completed = {
        (str(r["id"]), str(r["variant"]))
        for r in rows
        if pd.notna(r.get("score_fom5_v2"))
    }

    print(f"Checkpoint encontrado: {len(completed)} casos completos.")
else:
    rows = []
    completed = set()

for key, evaluation in calibration_outputs.items():
    if key not in completed:
        rows = [
            r for r in rows
            if (str(r.get("id")), str(r.get("variant"))) != key
        ]
        rows.append(evaluation)
        completed.add(key)

pd.DataFrame(rows).to_csv(
    CHECKPOINT,
    index=False,
    encoding="utf-8-sig",
)

total = len(old_df)
print(f"Completados antes del loop: {len(completed)}/{total}")
print(f"Pendientes: {total - len(completed)}/{total}")

for _, rec in tqdm(
    old_df.iterrows(),
    total=total,
    desc="FOM-5 local definitivo"
):
    key = (str(rec["id"]), str(rec["variant"]))

    if key in completed:
        continue

    item = GOLD_LOOKUP[rec["id"]]
    out = rec.to_dict()

    try:
        t0 = time.time()

        parsed, raw, in_tok, out_tok = judge_local(
            item,
            rec["candidate"],
        )

        elapsed = time.time() - t0
        agg = aggregate_v2(item, parsed)

        out.update(agg)
        out.update({
            "has_formulation_v2": bool(parsed["has_formulation"]),
            "fatal_errors_v2": json.dumps(parsed.get("fatal_errors", []), ensure_ascii=False),
            "requirements_v2": json.dumps(parsed["requirements"], ensure_ascii=False),
            "judge_scores_raw_v2": json.dumps(parsed["scores"], ensure_ascii=False),
            "judge_v2_raw": raw,
            "judge_v2_error": None,
            "judge_input_tokens": in_tok,
            "judge_output_tokens": out_tok,
            "judge_latency_sec": elapsed,
            "judge_model": JUDGE_MODEL,
        })

    except Exception as e:
        out.update({
            "score_fom5_v2": np.nan,
            "decision_domains_v2": np.nan,
            "objective_v2": np.nan,
            "constraints_logic_v2": np.nan,
            "algebra_validity_v2": np.nan,
            "generalization_v2": np.nan,
            "requirement_coverage_v2": np.nan,
            "objective_coverage": np.nan,
            "constraint_coverage": np.nan,
            "domain_coverage": np.nan,
            "has_formulation_v2": np.nan,
            "fatal_errors_v2": None,
            "requirements_v2": None,
            "judge_scores_raw_v2": None,
            "judge_v2_raw": None,
            "judge_v2_error": str(e),
            "judge_input_tokens": np.nan,
            "judge_output_tokens": np.nan,
            "judge_latency_sec": np.nan,
            "judge_model": JUDGE_MODEL,
        })

    rows = [
        r for r in rows
        if (str(r.get("id")), str(r.get("variant"))) != key
    ]
    rows.append(out)

    if pd.notna(out.get("score_fom5_v2")):
        completed.add(key)

    pd.DataFrame(rows).to_csv(
        CHECKPOINT,
        index=False,
        encoding="utf-8-sig",
    )

v2 = pd.DataFrame(rows)

order = {
    (str(rec["id"]), str(rec["variant"])): i
    for i, (_, rec) in enumerate(old_df.iterrows())
}

v2["_order"] = [
    order.get((str(rec["id"]), str(rec["variant"])), 9999)
    for _, rec in v2.iterrows()
]

v2 = (
    v2
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

v2.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
v2.to_json(FINAL_JSONL, orient="records", lines=True, force_ascii=False)

n_ok = int(v2["score_fom5_v2"].notna().sum())

print()
print("==============================")
print("RESULTADO")
print("==============================")
print(f"Correctas: {n_ok}/{total}")
print(f"Fallidas:  {total - n_ok}/{total}")
print("CSV:", FINAL_CSV)

display(v2[[
    "id",
    "variant",
    "score_fom5_v2",
    "requirement_coverage_v2",
    "has_formulation_v2",
    "judge_latency_sec",
    "judge_v2_error",
]])


## 8. Baseline oficial

Solo entran los 30 problemas del benchmark.  
El caso oftalmológico se reporta aparte y **no modifica la media oficial**.


In [ ]:
successful = v2["score_fom5_v2"].notna()

if successful.sum() != len(old_df):
    print(
        "ADVERTENCIA: hay evaluaciones fallidas. "
        "No congeles el baseline hasta obtener 32/32."
    )

baseline_v2 = v2[
    (v2["split"] == "benchmark")
    & (v2["variant"] == "baseline_direct_deterministic")
    & v2["score_fom5_v2"].notna()
].copy()

print("====================================")
print("BASELINE — FOM-5 v2 LOCAL")
print("====================================")

summary = {
    "n": len(baseline_v2),
    "mean_fom5": baseline_v2["score_fom5_v2"].mean(),
    "median_fom5": baseline_v2["score_fom5_v2"].median(),
    "std_fom5": baseline_v2["score_fom5_v2"].std(ddof=1),
    "min_fom5": baseline_v2["score_fom5_v2"].min(),
    "max_fom5": baseline_v2["score_fom5_v2"].max(),
    "mean_requirement_coverage": baseline_v2["requirement_coverage_v2"].mean(),
    "mean_judge_latency_sec": baseline_v2["judge_latency_sec"].mean(),
}

for key, value in summary.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.3f}")
    else:
        print(f"{key}: {value}")

print("\nPromedios por dimensión:")

dimension_cols = [
    "decision_domains_v2",
    "objective_v2",
    "constraints_logic_v2",
    "algebra_validity_v2",
    "generalization_v2",
]

display(
    baseline_v2[dimension_cols]
    .mean()
    .to_frame("mean")
)

if "difficulty" in baseline_v2.columns:
    print("\nPor dificultad:")
    display(
        baseline_v2
        .groupby("difficulty")["score_fom5_v2"]
        .agg(["count", "mean", "median"])
    )

print("\nCaso oftalmológico (fuera del promedio):")

display(
    v2[v2["id"] == "diagnostic_oftalmo"][[
        "variant",
        "score_fom5_v2",
        "decision_domains_v2",
        "objective_v2",
        "constraints_logic_v2",
        "algebra_validity_v2",
        "generalization_v2",
        "requirement_coverage_v2",
        "fatal_errors_v2",
    ]]
)


## 9. Descargar resultados

Cuando obtengas `32/32`, descarga `resultados_fom5_local.csv` y consérvalo como baseline congelado antes de implementar IR + compilador.


In [ ]:
if FINAL_CSV.exists():
    files.download(str(FINAL_CSV))
else:
    print("Aún no existe el resultado final.")


## Protocolo que queda congelado si la calibración pasa

```text
Benchmark oficial: 30 problemas
Modelo evaluado: Qwen/Qwen2.5-1.5B-Instruct
Judge offline: Qwen/Qwen3-14B, 4-bit NF4
Decodificación judge: determinista (do_sample=False)
Thinking del judge: desactivado cuando el chat template lo permite
Salida judge: JSON compacto
FOM-5 v2:
  15% decisiones/dominios
  20% objetivo
  35% restricciones/lógica
  15% validez algebraica
  15% generalización
```

Las futuras variantes (`IR + compilador`, `Best-of-N`, `Generate→Validate→Repair`) deben evaluarse con **este mismo protocolo**, sin modificar golds, requirements, prompt del judge, modelo judge ni ponderaciones.
